#### Definição das variáveis de interesse da separação por perfil de município

1. Eliminação de Multicolinearidade

Indicadores sociais possuem dependência entre si. Se você utilizar a porcentagem de famílias no Cadastro Único e a porcentagem recebendo Bolsa Família, estará dando um peso duplo ao mesmo fenômeno.

2. Análise de Componentes Principais (PCA)

3. Selecionar os mais fortes no ranking da PCA 

In [18]:
from pathlib import Path
import pandas as pd

base_path = Path.cwd().parent
file_path = base_path / "data" / "processed" / "Base_de_Dados_Processada.csv"

df = pd.read_csv(file_path)

df.head()

,População_2022,Pontuação Indice ODS 2025,Goal 1 Score,Goal 2 Score,Goal 3 Score,Goal 5 Score,Goal 6 Score,Goal 7 Score,Goal 8 Score,Goal 9 Score,...,SDG16_CRP,Normalizado (0-100): SDG16_CRP,SDG16_DRTHMN,Normalizado (0-100): SDG16_DRTHMN,SDG16_TRANSP,Normalizado (0-100): SDG16_TRANSP,SDG17_1_INVST_PB,Normalizado (0-100): SDG17_1_INVST_PB,SDG17_3_P_RC_TRB,Normalizado (0-100): SDG17_3_P_RC_TRB
0,21495.0,41.885064,53.769451,53.255278,66.451721,58.730307,20.177142,56.309444,46.940699,11.631238,...,85.714,85.714,42.857,42.857,66.667,66.667,13.216,0.000000,5.257,8.108054
1,96833.0,45.475390,54.219225,50.890756,65.677811,41.654565,72.408554,68.007222,49.281375,14.603964,...,85.714,85.714,57.143,57.143,66.667,66.667,85.642,1.133196,18.394,34.298246
2,5363.0,47.569518,49.228293,43.817398,89.494428,48.938353,44.025511,63.241944,54.613778,7.181381,...,85.714,85.714,42.857,42.857,66.667,66.667,909.852,38.715329,5.686,8.963317
3,86895.0,48.582323,50.752090,51.083977,73.026011,47.023041,66.630268,69.962222,45.581682,17.049953,...,85.714,85.714,57.143,57.143,50.000,50.000,328.073,12.187507,21.627,40.743620
4,15890.0,48.718897,56.552994,50.268036,79.958615,32.086768,65.794541,68.762222,52.890151,11.397874,...,100.000,100.000,42.857,42.857,66.667,66.667,246.352,8.461212,12.178,21.905901


####  Filtro de Multicolinearidade (Corte de variáveis com correlação > 0.80)

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

cols_to_keep = [c for c in df.columns if c.startswith('SDG') 
                and 'SDG4' not in c 
                and 'IDEB' not in c ]

df = df[cols_to_keep].copy()

corr_matrix = df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.80)]
X_filtered = df.drop(columns=to_drop)

print(f"Total de variáveis iniciais: {len(df.columns)}")
print(f"Total após remover redundantes (Correlação > 0.8): {len(X_filtered.columns)}")

Total de variáveis iniciais: 85
Total após remover redundantes (Correlação > 0.8): 77


#### Aplicação da análise de componentes principais 

In [20]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filtered)

# 5 principais (a definir)
pca = PCA(n_components=5) 
pca.fit(X_scaled)

# Extração da importância de cada vartiável 
loadings = pd.DataFrame(pca.components_.T, columns=['PC1', 'PC2', 'PC3', 'PC4', 'PC5'], index=X_filtered.columns)

### Resultados

In [21]:

print("\n--- AS VARIÁVEIS MAIS RELEVANTES PARA CLUSTERING ---")
print("\nVariáveis que lideram o Eixo 1 (Geralmente Eixo Social/Desigualdade):")
print(loadings['PC1'].abs().sort_values(ascending=False).head(3))

print("\nVariáveis que lideram o Eixo 2 (Geralmente Capacidade Institucional/Urbana):")
print(loadings['PC2'].abs().sort_values(ascending=False).head(3))

print("\nVariáveis que lideram o Eixo 3:")
print(loadings['PC3'].abs().sort_values(ascending=False).head(3))

print("\nVariáveis que lideram o Eixo 4:")
print(loadings['PC4'].abs().sort_values(ascending=False).head(3))

print("\nVariáveis que lideram o Eixo 5:")
print(loadings['PC5'].abs().sort_values(ascending=False).head(3))



--- AS VARIÁVEIS MAIS RELEVANTES PARA CLUSTERING ---

Variáveis que lideram o Eixo 1 (Geralmente Eixo Social/Desigualdade):
SDG1_1_P_F_CU      0.261178
SDG10_1_PREN20     0.259850
SDG1_4_R_1_4_SM    0.255672
Name: PC1, dtype: float64

Variáveis que lideram o Eixo 2 (Geralmente Capacidade Institucional/Urbana):
SDG17_3_P_RC_TRB    0.260455
SDG13_4_PREV        0.251372
SDG11_4_P_P_SUBN    0.233986
Name: PC2, dtype: float64

Variáveis que lideram o Eixo 3:
SDG15_2_FLOR      0.279753
SDG7_2_ENRG       0.255372
SDG7_3_VNLENER    0.250503
Name: PC3, dtype: float64

Variáveis que lideram o Eixo 4:
SDG5_7_JV_NENT      0.314241
SDG5_3_M_JV_NENT    0.280683
SDG8_2_OCP_INF      0.263622
Name: PC4, dtype: float64

Variáveis que lideram o Eixo 5:
SDG16_3_M_ARM_FG    0.464054
SDG16_1_OB_HM       0.414703
SDG10_3_RZHOM       0.336802
Name: PC5, dtype: float64


Explicação das variáveis de primeira posição em cada eixo:

- SDG1_1_P_F_CU: Famílias inscritas no Cadastro Único para programas sociais (%)
- SDG17_3_P_RC_TRB: Total de receitas municipais arrecadadas (%)
- SDG15_2_FLOR: Hectare de áreas florestadas e naturais por habitante
- SDG5_7_JV_NENT: Diferença percentual entre jovens mulheres e homens que não estudam e nem trabalham (p.p.)
- SDG16_3_M_ARM_FG: Mortes por armas de fogo (100 mil habitantes)
